---
toc: true
image: example.png
pub-info:
    abstract: |
        vidigi 2.0.0 adds a whole numbers-in/DataFrames-out analysis layer
        (`vidigi.analysis`) and a matching plotting layer (`vidigi.plots`), reachable
        either as free functions on any event log or as `TrialLogger`/`EventLogger`
        methods. This tours all of it: duration extraction, distributions, resource
        utilisation, warm-up and replication diagnostics, metric-vs-arrival-time, and
        a few smaller additions elsewhere in the logging, animation and resource APIs. Some of
        these already have their own deep-dive notebook - this gives each just enough
        room to be useful on its own, and links out for the rest of the story.
execute:
  enabled: true
---

# Feature Example: A Tour of vidigi 2.0.0's Analytics & Plotting Additions

Before 2.0.0, vidigi's non-animation output was four methods: `plot_entity_timeline`,
`generate_dfg`, `plot_metric_bar` (bare bars, no uncertainty), and `plot_queue_size`.
2.0.0 adds a proper analysis surface on top of the same event logs: duration
distributions, resource utilisation, confidence intervals across replications, a
warm-up diagnostic, a replication-count diagnostic, and a metric-vs-arrival-time
diagnostic - plus a few smaller quality-of-life additions to the logging and
animation APIs themselves.

Every `vidigi.plots` function is a thin wrapper over a matching `vidigi.analysis`
function that returns the numbers alone, for tables and reports; `TrialLogger` gets
one-line delegating methods for both, which is the route used throughout this
notebook. Three of these features - warm-up diagnostics, replication-count
diagnostics, and metric-vs-arrival-time - already have their own dedicated notebook,
so they get a short, real demonstration here with a link to the full treatment
rather than being re-explained from scratch. The animation picked up new arguments
too: [feat_animation_warm_up.ipynb](../feat_animation_warm_up/feat_animation_warm_up.ipynb)
covers discarding a warm-up period from an animation, and the `queue_direction`
section near the end here is the short version of
[feat_queue_direction.ipynb](../feat_queue_direction/feat_queue_direction.ipynb).

## Model setup

The same single-resource clinic model used by
[feat_warm_up.ipynb](../feat_warm_up/feat_warm_up.ipynb),
[feat_replication_analysis.ipynb](../feat_replication_analysis/feat_replication_analysis.ipynb)
and
[feat_metric_vs_arrival_time.ipynb](../feat_metric_vs_arrival_time/feat_metric_vs_arrival_time.ipynb) -
patients arrive, wait for one of 4 treatment cubicles, are treated, and leave. Reusing
it means every number below is directly comparable with those three notebooks (the
~78% cubicle utilisation quoted there is the same figure this notebook derives
independently in the resource-utilisation section).

In [ ]:
import random

import plotly.io as pio
import simpy
from sim_tools.distributions import Exponential, Lognormal

from vidigi import analysis
from vidigi.logging import EventLogger, TrialLogger
from vidigi.resources import VidigiStore
from vidigi.utils import EventPosition, create_event_position_df

pio.renderers.default = "notebook"

In [ ]:
class g:
    """
    Create a scenario to parameterise the simulation model

    Parameters:
    -----------
    random_number_set: int
        Set to control the initial seeds of each stream of pseudo
        random numbers used in the model.

    n_cubicles: int
        The number of treatment cubicles

    treat_mean, treat_var: float
        Mean and variance of the treatment duration distribution (Lognormal)

    arrival_rate: float
        Mean of the exponential inter-arrival time distribution

    sim_duration: int
        The number of time units the simulation will run for

    number_of_runs: int
        The number of replications
    """

    random_number_set = 42

    n_cubicles = 4
    treat_mean = 25
    treat_var = 5

    arrival_rate = 8

    sim_duration = 3000
    number_of_runs = 20

In [ ]:
class Patient:
    """Class defining details for a patient entity"""

    def __init__(self, p_id):
        self.id = p_id

In [ ]:
class Model:
    def __init__(self, run_number):
        self.env = simpy.Environment()
        self.run_number = run_number
        self.logger = EventLogger(env=self.env, run_number=self.run_number)
        self.patient_counter = 0
        self.init_distributions()
        self.init_resources()

    def init_distributions(self):
        self.patient_inter_arrival_dist = Exponential(
            mean=g.arrival_rate, random_seed=self.run_number * g.random_number_set
        )
        self.treat_dist = Lognormal(
            mean=g.treat_mean,
            stdev=g.treat_var,
            random_seed=self.run_number * g.random_number_set,
        )

    def init_resources(self):
        self.treatment_cubicles = VidigiStore(
            self.env, num_resources=g.n_cubicles, label="treatment_cubicle"
        )

    def generator_patient_arrivals(self):
        while True:
            self.patient_counter += 1
            p = Patient(self.patient_counter)
            self.env.process(self.attend_clinic(p))
            yield self.env.timeout(self.patient_inter_arrival_dist.sample())

    def attend_clinic(self, patient):
        self.logger.log_arrival(entity_id=patient.id)
        self.logger.log_queue(entity_id=patient.id, event="treatment_wait_begins")
        with self.treatment_cubicles.request() as req:
            treatment_resource = yield req
            self.logger.log_resource_use_start(
                entity_id=patient.id,
                event="treatment_begins",
                resource_id=treatment_resource.id,
                unique_resource_id=treatment_resource.unique_id,
            )
            yield self.env.timeout(self.treat_dist.sample())
            self.logger.log_resource_use_end(
                entity_id=patient.id,
                event="treatment_complete",
                resource_id=treatment_resource.id,
                unique_resource_id=treatment_resource.unique_id,
            )
        self.logger.log_departure(entity_id=patient.id)

    def run(self):
        self.env.process(self.generator_patient_arrivals())
        self.env.run(until=g.sim_duration)

In [ ]:
class Trial:
    def __init__(self):
        self.all_event_logs = []
        self.run_trial()

    def run_trial(self):
        for run in range(1, g.number_of_runs + 1):
            random.seed(run)
            my_model = Model(run)
            my_model.run()
            self.all_event_logs.append(my_model.logger)

In [ ]:
clinic_trial = Trial()
trial_logs = TrialLogger(
    clinic_trial.all_event_logs, scenario=g(), label="base scenario"
)
event_log = trial_logs.to_dataframe()
trial_logs.summary()

## Getting the raw numbers: `event_durations` / `get_event_durations`

Everything else in this notebook - distributions, bar charts, warm-up and
replication diagnostics, metric-vs-arrival-time - is built on one function:
`event_durations`, which pairs two events per entity and returns a duration, one row
per pairing, with no aggregation. `TrialLogger.get_event_durations` is the same thing
called on the trial's combined log. It replaces an older `pivot`-based calculation
that raised on any entity revisiting a step (a rework loop); this one supports that
via `match="first"|"last"|"occurrence"`.

In [ ]:
wait_durations = trial_logs.get_event_durations(
    "treatment_wait_begins", "treatment_begins"
)
wait_durations.head()

In [ ]:
wait_durations["duration"].describe()

`entity_id`, `run_number`, `pathway` and `occurrence` come along for free, and a
pairing with no matching second event (still queuing when the run ends) is kept with
`duration = NaN` rather than dropped, by default (`keep_incomplete=True`).

## One summary number, pooled or per-replication: `get_event_duration_stat` / `get_event_duration_ci`

`get_event_durations` gives every duration; often you want just one number.
`get_event_duration_stat` reduces them to a single statistic. Its default
`across="entities"` pools every entity's duration across every replication - one
big sample, run boundaries ignored - exactly as every prior release. `across="runs"`
(new in 2.0.0) instead computes the statistic within each run and averages those,
weighting every replication equally rather than by how many patients it happened to
see. `get_event_duration_ci` (also new) goes one step further, returning that
per-replication mean with a Student's t confidence interval across the 20 runs - the
headline "what is this number, and how sure are we" figure, and the numbers-only
twin of the `plot_metric_bar(across="runs", error_bars="ci")` bar in the next
section.

See [Choosing how to summarise across replications](/vidigi_docs/choosing_how_to_average.qmd)
for which one a given question wants, and
[feat_trial_logger.ipynb](../feat_trial_logger/feat_trial_logger.ipynb) for the full
treatment.

In [ ]:
pooled = trial_logs.get_event_duration_stat("treatment_wait_begins", "treatment_begins")
per_run = trial_logs.get_event_duration_stat(
    "treatment_wait_begins", "treatment_begins", across="runs"
)
ci = trial_logs.get_event_duration_ci("treatment_wait_begins", "treatment_begins")

print(f"pooled over all patients : {pooled}")
print(f"mean of per-run means    : {per_run}")
print(
    f"per-run mean with 95% CI : {ci.mean:.2f}  ({ci.lower:.2f} to {ci.upper:.2f}, n={ci.n})"
)

## Seeing the shape of a duration: `plot_duration_distribution`

A single statistic hides the shape - two systems with the same mean wait can look
very different once you see the spread. `plot_duration_distribution` draws it as a
histogram, box, violin, ECDF, or (given `split_by`) a ridgeline or heatmap comparing
many groups at once. All six `kind=` options, and `split_by="run"|"pathway"`, are
demonstrated in full in
[feat_trial_logger.ipynb](../feat_trial_logger/feat_trial_logger.ipynb) -
here's the one most useful for this model, a violin per run:

In [ ]:
trial_logs.plot_duration_distribution(
    "treatment_wait_begins",
    "treatment_begins",
    kind="violin",
    split_by="run",
    title="Treatment wait, by run",
)

## Uncertainty on a bar chart: `plot_metric_bar`'s `across=`/`error_bars=`

`plot_metric_bar` used to draw a bare bar - a single number, no sense of how much it
varies between replications. `across="runs"` computes the statistic separately
within each run first, then `error_bars="ci"` draws a confidence interval over those
per-run values (the only statistically valid unit to interval over - entities within
a run are correlated, replications are not). `show_runs=True` overlays each run's own
value as a point. Full coverage, including every `error_bars` option, in
[feat_trial_logger.ipynb](../feat_trial_logger/feat_trial_logger.ipynb):

In [ ]:
trial_logs.plot_metric_bar(
    [
        {
            "first_event": "treatment_wait_begins",
            "second_event": "treatment_begins",
            "label": "Treatment wait",
        }
    ],
    what="mean",
    across="runs",
    error_bars="ci",
    show_runs=True,
    title="Mean treatment wait, with 95% CI across runs",
    width=700,
)

## How busy was this resource?

Before 2.0.0 there was no way to answer this at all - the events were there
(`resource_use`/`resource_use_end` pairs on `resource_id`), but nothing computed busy
time or utilisation from them. Three new `vidigi.analysis` functions build on top of
`resource_use_intervals`, which pairs the raw `resource_use`/`resource_use_end` rows
into one interval per bout of use:

In [ ]:
intervals = analysis.resource_use_intervals(event_log)
intervals.head()

An entity still holding a resource when the trial's window ends is **censored**
by default (`unclosed="censor"`), not dropped - its interval is clipped to the window
end rather than discarded, since dropping it would understate utilisation exactly
when it matters most (a system that is still busy at the end of a run is
disproportionately a congested one). Here that affects a small minority of bouts:

In [ ]:
print(
    f"{intervals['censored'].sum()} of {len(intervals)} bouts were still open when their run ended"
)

`resource_occupancy_over_time` is the resource equivalent of `plot_queue_size`'s
data - how many units were busy at regular snapshots, computed exactly via a +1/-1
sweep over the intervals above rather than a per-snapshot scan:

In [ ]:
occupancy = analysis.resource_occupancy_over_time(event_log, every_x_time_units=50)
occupancy.head()

### `resource_utilisation`: four ways to supply capacity

Busy time and how many units were in use on average (`mean_in_use`) need no capacity
at all. `utilisation` (`mean_in_use / capacity`) does, and there are four ways to
supply it, in precedence order. All four resolve to the same answer here, since
they're describing the one real model:

- **A - an explicit dict**, the simplest route when you already know the numbers:

In [ ]:
ru_a = analysis.resource_utilisation(
    event_log, resource_capacities={"treatment_begins": 4}
)
ru_a["utilisation"].agg(["mean", "min", "max"])

- **B - a `scenario` plus a step-to-attribute mapping**, so capacity is read
  directly off the same scenario the model itself uses (no number to keep in
  sync by hand). `scenario` can be that parameter object *or* a plain dict
  (`scenario={"n_cubicles": 4}`) - the mapping's values are resolved as
  attributes or dict keys interchangeably:

In [ ]:
resource_map = {"treatment_begins": "n_cubicles"}
ru_b = analysis.resource_utilisation(event_log, scenario=g(), resource_map=resource_map)
ru_b["utilisation"].agg(["mean", "min", "max"])

- **C - a `scenario` plus an `event_position_df`**, reusing the same `resource=`
  column the animation functions already read capacity from - handy when one already
  exists for the animation and you don't want to write a second mapping. Pass such an
  `event_position_df` to an animation *without* a `scenario` and vidigi now warns
  rather than silently drawing no resource icons:

In [ ]:
event_position_df = create_event_position_df(
    [
        EventPosition(event="arrival", x=50, y=300, label="Arrival"),
        EventPosition(
            event="treatment_wait_begins", x=205, y=275, label="Waiting for Treatment"
        ),
        EventPosition(
            event="treatment_begins",
            x=205,
            y=175,
            label="Being Treated",
            resource="n_cubicles",
        ),
        EventPosition(event="depart", x=270, y=70, label="Exit"),
    ]
)
ru_c = analysis.resource_utilisation(
    event_log, scenario=g(), event_position_df=event_position_df
)
ru_c["utilisation"].agg(["mean", "min", "max"])

- **D - `capacity="infer"`**, for a log with no scenario at all (e.g. from `ciw` or a
  CSV): capacity is estimated as the number of distinct `resource_id`s seen for a
  step. It's always a **lower bound** - a unit that was never used is invisible - and
  always warns. It happens to recover the true capacity exactly here, because every
  one of the 4 cubicles gets used at some point across 3000 time units:

In [ ]:
ru_d = analysis.resource_utilisation(event_log, capacity="infer")
ru_d["capacity"].unique()

Across 20 replications, mean cubicle utilisation is a fairly stable ~78%
(bouncing between roughly 73% and 84% run-to-run) - the same figure
[feat_warm_up.ipynb](../feat_warm_up/feat_warm_up.ipynb) quotes for this model, derived
independently here rather than carried over.

`trial_logs` was built with `scenario=g()` attached (see *Carrying the scenario with
the trial* below), so the `TrialLogger` resource-utilisation methods from here on
read capacity straight from it - `resource_map` is all they need, no `scenario=` on
each call. The `vidigi.analysis` free functions above still take `scenario=`
explicitly; only the `TrialLogger` methods have somewhere to read it from.

`by="resource"` breaks utilisation down per physical unit instead of per step -
useful for spotting one persistently busier cubicle. It needs `resource_id` to be
unique across the whole log, not just within one step; `resource_col_name=None` on
the `TrialLogger` methods (used below) auto-detects `unique_resource_id` when it's
present, which is what `VidigiStore(..., label=...)` logs alongside the plain
`resource_id` used above - the collision-safe route, covered in full in
[feat_trial_logger.ipynb](../feat_trial_logger/feat_trial_logger.ipynb):

In [ ]:
by_resource = trial_logs.get_resource_utilisation(by="resource", resource_col_name=None)
by_resource.groupby("resource_id")["utilisation"].mean()

### The two charts

`plot_resource_utilisation` is the bar-chart counterpart to `resource_utilisation`
(mean across runs, CI error bar, `error_bars="ci"`/`show_runs=True` by default since
this is new rather than extracted from older bar-only behaviour), with a dashed
reference line at 100% - utilisation can never legitimately exceed it, so a bar
crossing it is diagnostic of a capacity or logging problem. `plot_resource_utilisation_over_time`
is the resource equivalent of `plot_queue_size`, one step function per run plus a
bold mean:

In [ ]:
fig = trial_logs.plot_resource_utilisation(resource_map=resource_map)
fig.update_layout(title="Treatment cubicle utilisation")
fig.show()

In [ ]:
fig = trial_logs.plot_resource_utilisation_over_time(
    every_x_time_units=25,
    as_proportion=True,
    resource_map=resource_map,
)
fig.update_layout(
    width=900,
    height=500,
    title="Treatment cubicle occupancy over time (as a proportion of capacity)",
)
fig.show()

## Occupancy at each step: `activity_occupancy_stats`

`plot_queue_size` and `resource_occupancy_over_time` both answer *how many entities
were at this step over time*. `activity_occupancy_stats` collapses that to one row
per step - the mean, minimum, maximum and median number present - for every queue
step and every resource step at once. `across_runs="average"` (below) reports the
figure expected per replication; `across_runs="pool"` takes one statistic over every
run and snapshot together, so the maximum becomes the worst queue seen in *any* run.

It is the slow one in this notebook - the queue figures are rebuilt the same way
`animate_activity_log` reconstructs each frame, once per run - so `every_x_time_units`
is worth turning up on a long trial.

In [ ]:
analysis.activity_occupancy_stats(event_log, every_x_time_units=50)

This table is what `EventLogger.generate_dfg(occupancy_metrics=True)` merges onto a
process map's nodes, so a directly-follows graph can show where the queue built up
and how close each resource ran to capacity - see
[feat_process_maps.ipynb](../feat_process_maps/process_maps.ipynb).

## Choosing a warm-up length: `plot_warm_up_diagnostic`

Welch's procedure, visualised: ensemble-average a series across replications, then
smooth it with several window widths, and read off where they agree it's flattened
out. Full derivation and the "trim to 500" recommendation for this exact model are in
[feat_warm_up.ipynb](../feat_warm_up/feat_warm_up.ipynb); here's the diagnostic itself,
on the treatment queue:

In [ ]:
fig = trial_logs.plot_warm_up_diagnostic(
    series="queue",
    event="treatment_wait_begins",
    every_x_time_units=25,
    windows=(5, 10, 20),
)
fig.update_layout(
    width=900, height=500, title="Welch's procedure: treatment queue length"
)
fig.show()

## Choosing a replication count: `plot_replication_analysis`

The cumulative mean and its confidence interval as replications accumulate, plus the
relative half-width underneath against a dashed 5% reference - the point where it
*stays* below that line is the recommended minimum replication count.
[feat_replication_analysis.ipynb](../feat_replication_analysis/feat_replication_analysis.ipynb)
shows this model needs far more than the 20 replications run here to actually
converge - the chart below shows exactly why that's true at 20 alone:

In [ ]:
fig = trial_logs.plot_replication_analysis("treatment_wait_begins", "treatment_begins")
fig.update_layout(width=900, height=650)
fig.show()

## Does it matter when you arrived? `plot_metric_vs_arrival_time`

A third question, distinct from both of the above: not "how many replications" or
"how much warm-up", but whether a metric drifts *within* one run depending on when
the entity that produced it arrived - a non-stationary arrival process, or a
time-of-day load effect. Full treatment, including `rolling_window` vs `rolling_time`
smoothing and `colour_by`, is in
[feat_metric_vs_arrival_time.ipynb](../feat_metric_vs_arrival_time/feat_metric_vs_arrival_time.ipynb):

In [ ]:
fig = trial_logs.plot_metric_vs_arrival_time(
    "treatment_wait_begins",
    "treatment_begins",
    rolling_time=150,
    marker_size=3,
)
fig.update_layout(width=900, height=500)
fig.show()

## Keeping an entity timeline: `plot_entity_timeline`'s new `return_fig=`

`EventLogger.plot_entity_timeline` has always shown one entity's own journey through
the model - useful for debugging a specific case. Until 2.0.0 it only ever called
`fig.show()` and returned `None`; there was no way to keep the figure to restyle or
export it. `return_fig=True` returns it instead:

In [ ]:
fig = clinic_trial.all_event_logs[0].plot_entity_timeline(5, return_fig=True)
fig.update_layout(title="Entity 5's journey through the clinic (run 1)")
fig.show()

# fig.write_image("entity_5_timeline.png")  # now possible, since fig is a real Figure

The default is still `False` - existing scripts that rely on `plot_entity_timeline`
displaying itself keep working unchanged. The default is planned to flip to `True` at
vidigi 3.0.

## Animating from a logger directly: `logger.animate_activity_log()` and `run_number`

The animation sections that follow build a one-run DataFrame with
`clinic_trial.all_event_logs[0].to_dataframe()` before calling the `animate_activity_log`
*function*. As of 2.0.0 that conversion is optional - an `EventLogger` or `TrialLogger`
can be animated directly, either passed to the function as `event_log=` or, new, by
calling `animate_activity_log()` (or `reshape_for_animations()`) *as a method on the
logger itself* - no import, no `.to_dataframe()`:

```python
# an EventLogger animates its own single run - event_position_df is the only argument it needs
clinic_trial.all_event_logs[0].animate_activity_log(event_position_df, ...)
```

For a `TrialLogger`, `run_number=` picks the replication, and the `scenario` attached at
construction (here `g()`, see *Carrying the scenario with the trial* below) is reused - so
the call below passes neither a DataFrame nor `scenario=`, and resource-availability icons
still appear:

In [ ]:
# No import, no .to_dataframe(), no scenario= - the TrialLogger has all three
trial_logs.animate_activity_log(
    event_position_df,
    run_number=1,
    every_x_time_units=50,
    limit_duration=1000,
    plotly_height=450,
    plotly_width=1000,
)

`run_number` applies only to a `TrialLogger` - passing a multi-run `TrialLogger` without
it, or `run_number` alongside a DataFrame or an `EventLogger`, raises a `ValueError`. The
`animate_activity_log(event_log=...)` function form accepts all three inputs too; a plain
DataFrame works exactly as before, which is what the animation cells below use so they can
also show the function itself.

## Carrying the scenario with the trial: `scenario=` and `label=`

`EventLogger` and `TrialLogger` take an optional `scenario=` - the parameter object
(or plain dict) the runs came from - and a human-readable `label=`. This trial was
built with both, right at the top of the notebook:

```python
trial_logs = TrialLogger(clinic_trial.all_event_logs, scenario=g(), label="base scenario")
```

`scenario` is the same object-or-dict shape the resource-utilisation helpers accept,
so once it is attached `get_resource_utilisation`, `plot_resource_utilisation` and
`plot_resource_utilisation_over_time` read capacity straight from it - which is why
those calls in the resource section passed only `resource_map`, never `scenario=g()`.
A per-call `scenario=` still overrides it. A `TrialLogger` built from `EventLogger`s
that each carry a `scenario` / `label` inherits them (warning if the runs disagree),
and `vidigi.ciw.event_logger_from_ciw_recs` / `trial_logger_from_ciw_recs` gained
matching `scenario=` / `label=` arguments.

Both are plain attributes on the logger:

In [ ]:
print("label:", trial_logs.label)
print(
    "n_cubicles, read back from the attached scenario:", trial_logs.scenario.n_cubicles
)

### Saving a finished trial: `to_pickle` / `read_pickle`

Twenty runs of a stochastic model isn't free to regenerate, and the numbers only
mean anything alongside the parameters that produced them. `to_pickle()` writes the
whole trial - every run's events, the attached `scenario`, the `label` - to a single
file; `TrialLogger.read_pickle()` brings it back as a fully working `TrialLogger`
(`EventLogger` has the same pair).

A logger built with `env=` (the normal simpy pattern) used to be unpicklable - the
live `simpy.Environment` holds generators. The `env` is now dropped on pickle, since
it is only read while logging, so what you reload is a complete, finished record.

In [ ]:
import tempfile
from pathlib import Path

pkl_path = Path(tempfile.mkdtemp()) / "base_scenario_trial.pkl"
trial_logs.to_pickle(pkl_path)
print(f"wrote {pkl_path.name}  ({pkl_path.stat().st_size / 1_000:.0f} kB)")

reloaded = TrialLogger.read_pickle(pkl_path)
print("reloaded:", reloaded.summary())

# a real TrialLogger - the scenario came back with it, so utilisation still
# resolves from resource_map alone, matching the figure from before the round trip
reloaded.get_resource_utilisation(resource_map=resource_map)["utilisation"].agg(
    ["mean", "min", "max"]
)

## Flipping a queue's build direction: `queue_direction`

Queues have always built out to the *left* of their `event_position_df` anchor, so
the front of the line sits at the bottom-right corner. `queue_direction="right"` -
on `animate_activity_log`, `generate_animation` and `generate_animation_df` - mirrors
that: the anchor becomes the bottom-left corner and the queue extends rightwards,
wrapped rows included. It reads better with entity emojis that face right, and the
in-service icons and stage labels move to match. A per-event `direction` column on
`event_position_df` (or `EventPosition(..., direction="right")`) overrides it for a
single stage.

The default `"left"` is a verified no-op - existing animations are unchanged.
[feat_queue_direction.ipynb](../feat_queue_direction/feat_queue_direction.ipynb) is
the full walkthrough, including the per-event override; here it is on run 1 of the
model above, reusing the `event_position_df` from the resource-utilisation section:

In [ ]:
from vidigi.animation import animate_activity_log

single_run_log = clinic_trial.all_event_logs[0].to_dataframe()

animate_activity_log(
    event_log=single_run_log,
    event_position_df=event_position_df,
    scenario=g(),
    every_x_time_units=50,
    limit_duration=1000,
    queue_direction="right",
    plotly_height=450,
    plotly_width=1000,
)

## Flipping an icon in place: `flip_entity_icons`

`queue_direction` mirrors the *layout* to suit an icon's facing direction. Sometimes
the layout isn't free to move - a fixed background image, say - but the icon still
faces the wrong way. `flip_entity_icons=True` mirrors the icon itself instead,
leaving every coordinate (and `queue_direction`) untouched; a per-event `flip_icons`
column on `event_position_df` (or `EventPosition(..., flip_icons=True)`) overrides
it for a single stage, the same way `direction` overrides `queue_direction`.

The default `False` is a verified no-op.
[feat_flip_entity_icons.ipynb](../feat_flip_entity_icons/feat_flip_entity_icons.ipynb)
is the full walkthrough, including the per-event override and getting the CSS to a
page that isn't a live notebook; here it is on the same run and layout as above:

In [ ]:
animate_activity_log(
    event_log=single_run_log,
    event_position_df=event_position_df,
    scenario=g(),
    every_x_time_units=50,
    limit_duration=1000,
    flip_entity_icons=True,
    plotly_height=450,
    plotly_width=1000,
)

## Revealing a hidden entity honestly: `step_snapshot_reveal_pop_in`

`step_snapshot_max` caps how many individual entities a queue snapshot draws before
collapsing the rest into a "+ N more" count. When a queue that has been over the cap
shrinks back under it, the entity that reappears has actually been waiting all
along - but Plotly can't tell that apart from a genuine new arrival, since both are
simply "a point that wasn't drawn last frame, now is": it flies in from the top-left
corner of the plot either way, making a queue that's just draining look like a
sudden rush of new arrivals. `step_snapshot_reveal_pop_in=True` fixes exactly that
case: an entity hidden behind the cap now pops in at its actual queue position
instead. A genuine new arrival still flies in - that's the useful cue for "just
joined the system" - only reveals from behind the cap are affected.

The default `False` is a verified no-op, and turning it on costs one extra invisible
row per reveal, not per hidden entity or per frame hidden.
[feat_gauge_only_animations.ipynb](../feat_gauge_only_animations/feat_gauge_only_animations.ipynb)
has the full before/after comparison, including a stress test with hundreds of
concurrent reveals; here it's the same run and layout as above, with
`step_snapshot_max` turned down low enough to trigger a few reveals in this window:

In [ ]:
animate_activity_log(
    event_log=single_run_log,
    event_position_df=event_position_df,
    scenario=g(),
    every_x_time_units=50,
    limit_duration=1000,
    step_snapshot_max=4,
    wrap_queues_at=4,
    step_snapshot_reveal_pop_in=True,
    plotly_height=450,
    plotly_width=1000,
)

## A different cap per step: `step_snapshot_max_overrides`

`step_snapshot_max` is a single number for the whole animation, which forces a
compromise: set it high enough to show a genuine bottleneck queue honestly and every
small queue also renders dozens of icons; set it low and the bottleneck is hidden
behind a "+ N more" label the moment it matters. `step_snapshot_max_overrides` (new
in 2.0.0) takes a `{event: cap}` dict that overrides the scalar for named events
only - `{"waiting_for_bed": 250}` shows one long queue in full while every other step
stays capped at `step_snapshot_max`. Any event not in the dict uses the scalar, so it
stays the fallback; a key that matches no event in the log raises a warning, so a
misspelt event name isn't silently ignored.

The default `None` is a verified no-op, and like `step_snapshot_max` itself the
argument is accepted by `reshape_for_animations` and `generate_animation_df` as well
as `animate_activity_log`. The clinic model here has only one queue, so this cell
uses a small purpose-built log with two: twelve patients waiting for triage and
twelve waiting for a bed. The scalar cap of 5 applies to triage; `wait_bed` is
overridden to 20 and so is drawn in full:

In [ ]:
import pandas as pd

two_queue_rows = []
for pid in range(1, 13):
    two_queue_rows += [
        (0, pid, "arrival_departure", "arrival"),
        (0, pid, "queue", "wait_triage"),
        (200, pid, "arrival_departure", "depart"),
    ]
for pid in range(101, 113):
    two_queue_rows += [
        (0, pid, "arrival_departure", "arrival"),
        (0, pid, "queue", "wait_bed"),
        (200, pid, "arrival_departure", "depart"),
    ]
two_queue_log = pd.DataFrame(
    two_queue_rows, columns=["time", "entity_id", "event_type", "event"]
)

two_queue_positions = create_event_position_df(
    [
        EventPosition(event="arrival", x=50, y=300, label="Arrival"),
        EventPosition(event="wait_triage", x=380, y=300, label="Waiting for Triage"),
        EventPosition(event="wait_bed", x=380, y=180, label="Waiting for a Bed"),
        EventPosition(event="depart", x=270, y=70, label="Exit"),
    ]
)

animate_activity_log(
    event_log=two_queue_log,
    event_position_df=two_queue_positions,
    every_x_time_units=20,
    limit_duration=180,
    step_snapshot_max=5,
    step_snapshot_max_overrides={"wait_bed": 20},
    wrap_queues_at=5,
    plotly_height=450,
    plotly_width=1000,
)

## Choosing where arrivals come from: `spawn_in_from_arrival`

A `depart` row in `event_position_df` makes every entity move to a set point before
it disappears. Arrivals now behave the same way. Previously a brand-new entity flew
in from the plot's top-left corner, because a point that wasn't in the previous
frame has no position for Plotly to move it *from*; `spawn_in_from_arrival` (new in
2.0.0, **default `True`**) gives it one - the `EventPosition(event="arrival", ...)`
anchor - so a new entity glides in from there instead, the arrival-side mirror of
`depart`. It works by slipping the entity in at the anchor one snapshot early
(invisibly, then visibly), so there is a real prior position to animate the move
from.

This is a **breaking change** on the defaults: an animation whose layout gives
`"arrival"` a position - as every example in these docs does - now has its new
entities slide in from there. Pass `spawn_in_from_arrival=False` for the old
top-left fly-in. Only entities that arrive at least two snapshots in are affected;
anyone already in the system when the animation opens has no earlier frame to enter
from and still flies in. It's independent of `step_snapshot_reveal_pop_in` above,
which does the same job for entities re-emerging from behind a `+ N more` label.

Here the `arrival` anchor is moved to the top-right corner and labelled "Entrance",
so the effect is unmistakable - patients slide in from the entrance rather than the
corner of the plot:

In [ ]:
spawn_position_df = event_position_df.copy()
spawn_position_df.loc[spawn_position_df["event"] == "arrival", ["x", "y", "label"]] = [
    320,
    320,
    "Entrance",
]

animate_activity_log(
    event_log=single_run_log,
    event_position_df=spawn_position_df,
    scenario=g(),
    every_x_time_units=50,
    limit_duration=1000,
    spawn_in_from_arrival=True,
    plotly_height=450,
    plotly_width=1000,
)

## Beyond emoji: `entity_icon_font`, `entity_colour_by`, `resource_icon`

Every entity icon has, until now, been an emoji - and being *colour* fonts,
emoji ignore `textfont.color` entirely, so an entity could never be coloured
by anything. `entity_icon_font` switches the icon to an icon font (Font
Awesome, Bootstrap Icons, Material Symbols, or any font already on the page)
instead - thousands of monochrome glyphs, which `entity_colour_by` can then
colour meaningfully. Below, patients are coloured by the specific treatment
cubicle (`resource_id`) treating them - the same physical units
`resource_utilisation` measured earlier in this notebook - and the cubicles
themselves get a small bed icon via `resource_icon`. That's a field set per
event position (here via the `resource_icon` column), overriding
`custom_resource_icon` for that one stage, so different resource stages can
each carry their own icon; the value is a plain glyph or, as here, an image
(`bed_icon.svg`, drawn via `layout.images` rather than as text).

`entity_icon_font` / `resource_icon_font` need **plotly >= 5.23.0** (an icon
font resolves a numeric weight, which older plotly rejects); `pip install -U
plotly` if the cell below raises a `'weight' property` error. Emoji
animations are unaffected.

[feat_custom_icons.ipynb](../feat_custom_icons/feat_custom_icons.ipynb) is the
full derivation, including the four confirmed Plotly bugs found and worked
around along the way. That notebook also covers `entity_annotation_by`: once
an icon carries baked-in extra text (a running length-of-stay figure, say),
combining that with `flip_entity_icons`/`entity_icon_font` needs a second,
independently-styled trace rather than more text appended onto the icon
itself, since Plotly gives a single icon's `<text>` node one font and one
transform for the whole node - see the ["Annotating an icon with extra
text"](/vidigi_docs/customising_animations.qmd#annotating-an-icon-with-extra-text)
section of the docs for the full trade-off against appending.


In [ ]:
icon_position_df = event_position_df.copy()
icon_position_df.loc[
    icon_position_df["event"] == "treatment_begins", "resource_icon"
] = "bed_icon.svg"

animate_activity_log(
    event_log=single_run_log,
    event_position_df=icon_position_df,
    scenario=g(),
    every_x_time_units=50,
    limit_duration=1000,
    entity_icon_font="font-awesome",
    custom_entity_icon_list=[""],  # fa-walking
    entity_colour_by="resource_id",
    entity_colour_map={
        "1.0": "crimson",
        "2.0": "steelblue",
        "3.0": "seagreen",
        "4.0": "goldenrod",
        "nan": "lightgrey",  # not currently in treatment
    },
    gap_between_resources=30,
    plotly_height=450,
    plotly_width=1000,
)

## Automatic resource-use logging: `VidigiStore(..., logger=...)`

The model above logs every `resource_use`/`resource_use_end` event by hand, bracketing
`request()` with `log_resource_use_start`/`log_resource_use_end` calls. 2.0.0 adds an
opt-in `logger=` constructor parameter to `VidigiStore`/`VidigiPriorityStore` that
does this automatically: pass `entity_id=` (and optionally `start_event=`/`end_event=`)
into `request()`, and the start event is logged the moment the request is actually
granted, the end event the moment the resource is released - no manual logging calls
at all. The same model, rewritten:

In [ ]:
class AutoLoggingModel:
    def __init__(self, run_number):
        self.env = simpy.Environment()
        self.run_number = run_number
        self.logger = EventLogger(env=self.env, run_number=self.run_number)
        self.patient_counter = 0
        self.patient_inter_arrival_dist = Exponential(
            mean=g.arrival_rate, random_seed=run_number * g.random_number_set
        )
        self.treat_dist = Lognormal(
            mean=g.treat_mean,
            stdev=g.treat_var,
            random_seed=run_number * g.random_number_set,
        )
        # logger= is the only change here - VidigiStore now knows where to log to.
        self.treatment_cubicles = VidigiStore(
            self.env,
            num_resources=g.n_cubicles,
            label="treatment_cubicle",
            logger=self.logger,
        )

    def generator_patient_arrivals(self):
        while True:
            self.patient_counter += 1
            p = Patient(self.patient_counter)
            self.env.process(self.attend_clinic(p))
            yield self.env.timeout(self.patient_inter_arrival_dist.sample())

    def attend_clinic(self, patient):
        self.logger.log_arrival(entity_id=patient.id)
        self.logger.log_queue(entity_id=patient.id, event="treatment_wait_begins")
        # entity_id= (and the event names) are all request() needs to auto-log both
        # resource_use and resource_use_end - no log_resource_use_start/_end calls.
        with self.treatment_cubicles.request(
            entity_id=patient.id,
            start_event="treatment_begins",
            end_event="treatment_complete",
        ) as req:
            yield req
            yield self.env.timeout(self.treat_dist.sample())
        self.logger.log_departure(entity_id=patient.id)

    def run(self):
        self.env.process(self.generator_patient_arrivals())
        self.env.run(until=g.sim_duration)

In [ ]:
random.seed(1)
auto_model = AutoLoggingModel(1)
auto_model.run()

auto_intervals = analysis.resource_use_intervals(auto_model.logger.to_dataframe())
auto_intervals.head()

Run 1 uses the same seeds as run 1 of the hand-logged model above, so the two should
agree exactly - same start/end times, same `resource_id`s, for every bout. Comparing
them needs one extra thing pinned down first: `resource_use_intervals` resolves its
analysis window's end from the *latest time seen in the log it's given* unless
`limit_duration=` is passed explicitly, and the single run's own log genuinely ends
earlier (its last event) than the 20-run trial's combined log (the latest event
*anywhere* in the trial) - so the still-open bout at the end of run 1 would otherwise
get censored against two different window ends. Pin `limit_duration=g.sim_duration`
on both sides for a fair, exact comparison:

In [ ]:
manual_run_1 = analysis.resource_use_intervals(event_log, limit_duration=g.sim_duration)
manual_run_1 = manual_run_1[manual_run_1["run_number"] == 1].reset_index(drop=True)

auto_run_1 = analysis.resource_use_intervals(
    auto_model.logger.to_dataframe(), limit_duration=g.sim_duration
).reset_index(drop=True)

manual_run_1.drop(columns="run_number").equals(auto_run_1.drop(columns="run_number"))

`logger=` changes nothing about what gets logged, only how much code it takes to log
it.

## Inspecting a pool like a resource: `.count`, `.capacity`, `.n_waiting`

`VidigiStore` and `VidigiPriorityStore` are stores, but a fixed pool built with
`num_resources=` is really being used as a stand-in for `simpy.Resource`. 2.0.0 adds the
read-only properties you would reach for out of that habit:

- `.count` - units currently in use (checked out of the pool)
- `.n_waiting` - requests queued for a unit (the store analog of `len(simpy.Resource.queue)`)
- `.num_resources` - the pool size: the running total passed to `num_resources=` /
  `populate()`, later top-up `populate()` calls included

`.count` is derived from how many units are currently sitting in the store rather than
tracked per request, so it stays correct through `filter_fn` requests, reneging via
`cancel_get`, and `VidigiPriorityStore`'s direct holder-to-waiter handoff, with no
bookkeeping of your own. The invariant `0 <= count <= num_resources` is the same one simpy
keeps between `Resource.count` and `Resource.capacity`.

Everything in this section applies identically to `VidigiPriorityStore`.

In [ ]:
idle_pool = VidigiStore(
    simpy.Environment(), num_resources=g.n_cubicles, label="treatment_cubicle"
)
print(f"capacity      : {idle_pool.capacity}")
print(f"num_resources : {idle_pool.num_resources}")
print(f"count         : {idle_pool.count}")
print(f"n_waiting     : {idle_pool.n_waiting}")

### `.capacity` now returns the pool size

**Breaking change in 2.0.0** ([#87](https://github.com/Bergam0t/vidigi/issues/87)):
`.capacity` used to return the underlying container's item limit - `float("inf")` by
default, useless as a resource capacity. For a pool built with `num_resources=` /
`populate()` and no explicit `capacity=` it now returns the pool size, mirroring
`simpy.Resource.capacity` and equal to `.num_resources`. A store built with an explicit
`capacity=`, and a bare `VidigiStore(env)`, are unchanged:

In [ ]:
print("pooled  :", VidigiStore(simpy.Environment(), num_resources=4, label="bed").capacity)
print("bare    :", VidigiStore(simpy.Environment()).capacity)
print("explicit:", VidigiStore(simpy.Environment(), num_resources=4, label="bed", capacity=10).capacity)

### Watching occupancy during a run

Because `.count` and `.n_waiting` read live state, a monitoring process can sample them
exactly the way one would sample `simpy.Resource.count` - the in-simulation equivalent of
the `resource_occupancy_over_time` / `plot_queue_size` reconstructions earlier in this
notebook, which recover the same numbers from the event log after the fact. Here that
monitor runs alongside run 1 of the clinic model:

In [ ]:
import pandas as pd

monitored = Model(1)
pool_samples = []


def monitor_pool(env, store):
    while True:
        pool_samples.append(
            {"time": env.now, "in_use": store.count, "queued": store.n_waiting}
        )
        yield env.timeout(50)


monitored.env.process(monitor_pool(monitored.env, monitored.treatment_cubicles))
random.seed(1)
monitored.run()

pool_samples = pd.DataFrame(pool_samples)
pool_samples["in_use"].agg(["mean", "min", "max"])

Mean cubicle occupancy of ~3.07 of 4, about 77%, is in line with the utilisation
section above - derived here live rather than from the log. `in_use` never exceeds
the capacity of 4, the `0 <= count <= num_resources` invariant holding throughout
the run:

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(
    x=pool_samples["time"], y=pool_samples["in_use"],
    name="cubicles in use", line_shape="hv",
)
fig.add_scatter(
    x=pool_samples["time"], y=pool_samples["queued"],
    name="patients queued", line_shape="hv",
)
fig.add_hline(y=g.n_cubicles, line_dash="dash", annotation_text="capacity")
fig.update_layout(
    width=900, height=400,
    title="Cubicle pool: .count and .n_waiting through run 1",
    xaxis_title="time", yaxis_title="patients",
)
fig.show()

### Returning too many units now raises

**Breaking change in 2.0.0**: a `simpy.Resource` never lets you release more units than it
has. A `VidigiStore` pool used to accept an accidental extra `put()` - a unit returned
twice, or one from another pool - and grow silently past `num_resources`, surfacing only
later as a `.count` `RuntimeError`. That return now raises `ValueError` at the call site:

In [ ]:
env = simpy.Environment()
tills = VidigiStore(env, num_resources=2, label="till")
held = [tills.get_direct(), tills.get_direct()]
env.run()

for got in held:
    tills.put(got.value)  # both units back - the pool is now full

try:
    tills.put(held[0].value)  # the same unit a second time
except ValueError as err:
    print(err)

Pass `strict_capacity=False` to opt out - for a genuinely elastic pool meant to grow via
raw `put()`. `.count` can then no longer be trusted (the pool size is no longer fixed) and
raises its `RuntimeError` once the store holds more units than were populated:

In [ ]:
env = simpy.Environment()
elastic = VidigiStore(env, num_resources=2, label="till", strict_capacity=False)
held = [elastic.get_direct(), elastic.get_direct()]
env.run()
for got in held:
    elastic.put(got.value)
elastic.put(held[0].value)  # allowed - the pool grows to 3 units

print("units in pool:", len(elastic.items))
try:
    elastic.count
except RuntimeError as err:
    print(err)

The same `RuntimeError` is why `populate_store()` - the old free function for seeding a
store - is **deprecated in 2.0.0 and removed in 3.0**: a pool it fills is invisible to
`.count` / `.num_resources`, and the `num_resources=` constructor argument and
`.populate()` method now cover the same job. Build the pool with
`VidigiStore(env, num_resources=N, label=...)`, as every cell above does, or
`store.populate(N, label=...)` to top an existing one up.

## Closing note

Everything above is reachable two ways: as a free function in `vidigi.analysis`/
`vidigi.plots` (works on any event log - `vidigi.ciw`, a CSV, a hand-built frame, not
just `EventLogger`/`TrialLogger`), or as a one-line delegating method on `TrialLogger`
(used throughout this notebook, since it's the route most modellers will actually
use).

If your model is built with [`ciw`](https://ciw.readthedocs.io) rather than SimPy,
that second route is open to you too: `vidigi.ciw` gained
`event_logger_from_ciw_recs(recs, node_name_list=...)` and
`trial_logger_from_ciw_recs(list_of_recs, node_name_list=...)` in 2.0.0, which build a
populated `EventLogger` / `TrialLogger` straight from `Simulation.get_all_records()`
output - the same conversion as the long-standing `event_log_from_ciw_recs`, but
returning the logging object instead of a bare DataFrame. Every `TrialLogger` method
demonstrated above then works on a `ciw` trial unchanged. See the
[ciw examples](../example_10_advanced_ciw/ex_10_ciw.ipynb) for the conversion itself.

For the three diagnostics given only a brief treatment here, the full story is
one click away:

- [feat_warm_up.ipynb](../feat_warm_up/feat_warm_up.ipynb) - how much of a run to discard
- [feat_replication_analysis.ipynb](../feat_replication_analysis/feat_replication_analysis.ipynb) - how many replications to run
- [feat_metric_vs_arrival_time.ipynb](../feat_metric_vs_arrival_time/feat_metric_vs_arrival_time.ipynb) - whether a metric drifts within a run
- [feat_trial_logger.ipynb](../feat_trial_logger/feat_trial_logger.ipynb) - every `plot_duration_distribution` kind, every `plot_metric_bar`/`plot_resource_utilisation` error-bar option, and the `resource_id` collision problem `label=` exists to prevent